## التقييم النهائي لأداء نظام استرجاع المعلومات على LoTTE

هذا الـ Notebook مخصص لإجراء التقييم النهائي على مجموعة بيانات `lotte/lifestyle/dev` باستخدام النماذج المُعدة مسبقاً.

**المهام:**
1.  تحميل الاستعلامات (queries) وأحكام الصلة (qrels) من ملف `qas.search.jsonl`.
2.  إجراء تقييم شامل (MAP, MRR, P@10, R@10) لكل نموذج من النماذج.
3.  عرض النتائج النهائية في جدول للمقارنة.

### الخطوة 1: الإعدادات العامة وتحميل المكتبات

In [1]:
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
import sys
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns

# --- الإعدادات التي يمكنك تغييرها ---
# اسم مجموعة البيانات التي سيتم تقييمها (يجب أن يتطابق مع اسم المجلد في مسار النماذج)
DATASET_NAME = 'lotte/lifestyle/dev'
DATASET_SPLIT = 'lotte/lifestyle/dev' # يستخدم لإرساله للـ API

# المسار إلى ملف الاستعلامات الخاص بـ LoTTE
# تأكد من أن هذا المسار صحيح في بيئتك
LOTTE_QUERIES_FILE = '/home/hussam/.ir_datasets/lotte/lotte_extracted/lotte/lifestyle/dev/qas.search.jsonl'

# عدد الاستعلامات التي سيتم تقييمها (None لتقييم جميع الاستعلامات)
QUERY_SAMPLE_SIZE = None

# --- إعدادات النظام (لا تقم بتغييرها إلا إذا كنت تعرف ماذا تفعل) ---
SEARCH_API_URL = "http://127.0.0.1:8003/search/" # تأكد من أن البورت صحيح
MODELS_TO_EVALUATE = ['tfidf', 'bm25', 'bert', 'hybrid']

# أفضل المعاملات لنموذج BM25 التي تم التوصل إليها مسبقاً
BEST_K1 = 1.6
BEST_B = 0.75

### الخطوة 2: تحميل الاستعلامات وأحكام الصلة (Qrels) من LoTTE

In [2]:
queries = {}
qrels = {}

print(f"Loading queries and qrels from: {LOTTE_QUERIES_FILE}")
try:
    with open(LOTTE_QUERIES_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            # استخدام qid كسلسلة نصية لضمان التوافق
            query_id = str(data['qid'])
            query_text = data['query']
            
            # تحويل PIDs (معرفات المستندات ذات الصلة) إلى مجموعة من السلاسل النصية
            relevant_pids = {str(pid) for pid in data.get('answer_pids', [])}

            queries[query_id] = query_text
            if relevant_pids:
                qrels[query_id] = relevant_pids
                
    print(f"Successfully loaded {len(queries)} queries and {len(qrels)} qrels.")
except FileNotFoundError:
    print(f"ERROR: File not found at {LOTTE_QUERIES_FILE}")
    print("Please make sure the path is correct and the LoTTE dataset is available.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

# تحديد قائمة الاستعلامات التي سيتم تشغيل التقييم عليها
query_ids_with_rels = [qid for qid in queries.keys() if qid in qrels and qrels[qid]]
if QUERY_SAMPLE_SIZE:
    query_ids_to_run = query_ids_with_rels[:QUERY_SAMPLE_SIZE]
else:
    query_ids_to_run = query_ids_with_rels

print(f"Running evaluation on {len(query_ids_to_run)} queries that have relevance judgments.")

Loading queries and qrels from: /home/hussam/.ir_datasets/lotte/lotte_extracted/lotte/lifestyle/dev/qas.search.jsonl
Successfully loaded 417 queries and 417 qrels.
Running evaluation on 417 queries that have relevance judgments.


### الخطوة 3: تعريف دوال التقييم

In [3]:
def calculate_metrics(retrieved_doc_ids, relevant_doc_ids):
    """يحسب مقاييس AP, RR, P@10, R@10 لاستعلام واحد."""
    if not retrieved_doc_ids or not relevant_doc_ids:
        return {'AP': 0.0, 'RR': 0.0, 'P@10': 0.0, 'R@10': 0.0}
    
    hits = 0
    sum_precision = 0.0
    rr = 0.0
    first_hit_found = False
    
    for i, doc_id in enumerate(retrieved_doc_ids):
        if doc_id in relevant_doc_ids:
            if not first_hit_found:
                rr = 1 / (i + 1)
                first_hit_found = True
            hits += 1
            sum_precision += hits / (i + 1)
            
    ap = sum_precision / len(relevant_doc_ids)
    
    # حساب P@10 و R@10
    retrieved_at_10 = set(retrieved_doc_ids[:10])
    hits_at_10 = len(retrieved_at_10.intersection(relevant_doc_ids))
    p10 = hits_at_10 / 10.0
    r10 = hits_at_10 / len(relevant_doc_ids)
    
    return {'AP': ap, 'RR': rr, 'P@10': p10, 'R@10': r10}

def evaluate_model(query_ids, model_type):
    """يشغل التقييم لنموذج معين عبر جميع الاستعلامات."""
    all_metrics = []
    pbar = tqdm(query_ids, desc=f"Evaluating {model_type}", ncols=100)
    
    for query_id in pbar:
        query_text = queries.get(query_id)
        relevant_docs = qrels.get(query_id, set())
        
        if not query_text or not relevant_docs:
            continue
            
        # إعداد الطلب إلى API الخاص بالبحث
        payload = {
            "dataset_name": DATASET_NAME,
            "query": query_text,
            "model_type": model_type,
            "top_k": 100, # استرجاع أفضل 100 مستند للتقييم
            "k1": BEST_K1,
            "b": BEST_B,
            "enable_ner_reranking": False,
            "hybrid_bm25_weight": 0.1
        }
        
        try:
            response = requests.post(SEARCH_API_URL, json=payload, timeout=60)
            response.raise_for_status() # يثير خطأ في حالة فشل الطلب
            results = response.json().get('results', [])
            retrieved_doc_ids = [str(res['doc_id']) for res in results]
            metrics = calculate_metrics(retrieved_doc_ids, relevant_docs)
            all_metrics.append(metrics)
        except requests.exceptions.RequestException as e:
            print(f"\nAPI request failed for query {query_id}: {e}")
            # في حالة الفشل، نعتبر أن النموذج لم يسترجع أي شيء
            metrics = calculate_metrics([], relevant_docs)
            all_metrics.append(metrics)
            
    # حساب المتوسط لجميع المقاييس
    if all_metrics:
        return pd.DataFrame(all_metrics).mean().to_dict()
    else:
        return {'AP': 0, 'RR': 0, 'P@10': 0, 'R@10': 0}

### الخطوة 4: تنفيذ التقييم النهائي وعرض النتائج

In [4]:
if not query_ids_to_run:
    print("No queries to evaluate. Please check the data loading step.")
else:
    print(f"--- Running Final Evaluation on {len(query_ids_to_run)} Queries ---")
    final_results_list = []

    for model_type in MODELS_TO_EVALUATE:
        # استدعاء دالة التقييم بالمتغيرات الصحيحة
        metrics = evaluate_model(query_ids_to_run, model_type)
        metrics['Model'] = model_type
        final_results_list.append(metrics)

    # تحويل النتائج إلى DataFrame وتغيير أسماء الأعمدة
    final_df = pd.DataFrame(final_results_list).rename(columns={'AP': 'MAP', 'RR': 'MRR'})

    # عرض النتائج في جدول واحد نهائي
    results_df = final_df.set_index('Model')[['MAP', 'MRR', 'P@10', 'R@10']]

    print(f"\n--- Final Evaluation Results for '{DATASET_SPLIT}' ---")
    display(results_df.style.format('{:.4f}')\
                          .background_gradient(cmap='viridis', axis=0)\
                          .set_caption("مقارنة أداء النماذج الأساسية"))

--- Running Final Evaluation on 417 Queries ---


Evaluating tfidf:  11%|████▊                                       | 46/417 [00:06<00:45,  8.08it/s]


API request failed for query 44: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8003/search/


Evaluating tfidf:  22%|█████████▋                                  | 92/417 [00:13<00:21, 15.46it/s]


API request failed for query 90: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8003/search/


...

Evaluating tfidf:  99%|██████████████████████████████████████████▌| 413/417 [00:53<00:00,  9.92it/s]


API request failed for query 413: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8003/search/


Evaluating hybrid: 100%|██████████████████████████████████████████| 417/417 [02:13<00:00,  3.11it/s]


--- Final Evaluation Results for 'lotte/lifestyle/dev' ---


,MAP,MRR,P@10,R@10
Model,,,,
tfidf,0.0459,0.1057,0.0242,0.0822
bm25,0.0486,0.1076,0.0261,0.0877
bert,0.4028,0.5856,0.1556,0.5428
hybrid,0.4053,0.5904,0.1561,0.5442
